# 02 — Building the content engine

*Applied Labs · Domain: Prompt Engineering · Advanced · ~40 min*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI2026/castalia/blob/main/labs/07_content_seo_engine/02_build.ipynb)

**Prerequisites:** `01_architecture.ipynb`

Time to build. We'll implement every stage of the content engine:
brand voice library → topic research → brief generation → article
writing (OpenAI) → SEO optimization → quality evaluation — then run
the full pipeline end-to-end on three topics.

> **Learning goals**
> 1. Build a reusable brand voice library with two distinct profiles.
> 2. Implement a topic research agent with structured output.
> 3. Generate detailed content briefs from research.
> 4. Write full articles via OpenAI with brand voice control.
> 5. Score SEO compliance and content quality automatically.
> 6. Run the end-to-end pipeline with revision loops.

## 0 · Setup

In [ ]:
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "textstat>=0.7.0" "chromadb>=0.4.0"

In [ ]:
import os
import json
import re
import time
from typing import Any

import numpy as np
import textstat
from openai import OpenAI

OPENAI_API_KEY: str | None = os.getenv("OPENAI_API_KEY")
client: OpenAI | None = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
MODEL: str = "gpt-4o-mini"

print(f"OpenAI key configured: {OPENAI_API_KEY is not None}")
print(f"Model: {MODEL}")
print("Environment ready ✓")

## 1 · Brand voice library

We define **two distinct profiles** — a technical B2B SaaS voice and a
friendly consumer voice. Each profile contains tone descriptors,
vocabulary preferences, persona definition, and an example paragraph
that serves as a few-shot style anchor.

In [ ]:
# Brand voice profiles
VOICE_LIBRARY: dict[str, dict[str, Any]] = {
    "technical_b2b": {
        "name": "Technical B2B SaaS",
        "tone": ["authoritative", "precise", "data-driven", "professional"],
        "vocabulary": {
            "prefer": ["leverage", "optimize", "infrastructure", "scalable",
                       "enterprise-grade", "benchmark", "throughput"],
            "avoid": ["easy", "simple", "just", "basically", "obviously"],
        },
        "persona": (
            "Senior engineering lead writing for CTOs, VP Engineering, "
            "and platform teams at Series B+ companies."
        ),
        "sentence_style": "Concise, technical, with data points where possible.",
        "example_paragraphs": [
            (
                "Horizontal pod autoscaling in Kubernetes leverages real-time "
                "metric streams to dynamically adjust replica counts. Our "
                "benchmarks across 14 production clusters show a 37% reduction "
                "in compute waste when combining HPA with custom Prometheus "
                "metrics — without sacrificing p99 latency targets."
            ),
            (
                "The migration from monolithic deployments to microservices "
                "introduces non-trivial observability challenges. Organizations "
                "that invest in distributed tracing infrastructure early see "
                "a measurable reduction in mean-time-to-resolution, with our "
                "data indicating a 4.2× improvement for teams using OpenTelemetry."
            ),
        ],
    },
    "friendly_consumer": {
        "name": "Friendly Consumer Brand",
        "tone": ["conversational", "encouraging", "clear", "warm"],
        "vocabulary": {
            "prefer": ["easy", "simple", "step-by-step", "you'll love",
                       "no worries", "here's the thing", "game-changer"],
            "avoid": ["leverage", "optimize", "infrastructure", "enterprise",
                      "scalable", "throughput"],
        },
        "persona": (
            "Helpful friend who makes complex topics approachable. "
            "Writing for non-technical professionals and curious beginners."
        ),
        "sentence_style": "Short sentences, questions, analogies, encouraging tone.",
        "example_paragraphs": [
            (
                "Ever feel like your cloud bill is out of control? You're not "
                "alone! The good news is, Kubernetes has a built-in trick called "
                "autoscaling that can save you serious money. Think of it like a "
                "smart thermostat for your servers — it dials resources up when "
                "things get busy and back down when they don't."
            ),
            (
                "Setting up your first content calendar doesn't have to be "
                "overwhelming. Here's the thing: start small. Pick three topics "
                "you know inside out, schedule one post per week, and build from "
                "there. You'll be surprised how quickly the momentum builds!"
            ),
        ],
    },
}


def get_voice_prompt(voice_name: str) -> str:
    """Build a system prompt fragment from a voice profile.

    Parameters
    ----------
    voice_name : key into VOICE_LIBRARY

    Returns
    -------
    str – formatted voice instructions for the system prompt.
    """
    v = VOICE_LIBRARY[voice_name]
    examples = "\n\n".join(f'"{p}"' for p in v["example_paragraphs"])
    return (
        f"BRAND VOICE: {v['name']}\n"
        f"Tone: {', '.join(v['tone'])}\n"
        f"Persona: {v['persona']}\n"
        f"Sentence style: {v['sentence_style']}\n"
        f"Preferred words: {', '.join(v['vocabulary']['prefer'])}\n"
        f"Words to avoid: {', '.join(v['vocabulary']['avoid'])}\n\n"
        f"Style examples:\n{examples}"
    )


# Display both profiles
for name, profile in VOICE_LIBRARY.items():
    print(f"━━━ {profile['name']} ━━━")
    print(f"  Tone:    {', '.join(profile['tone'])}")
    print(f"  Persona: {profile['persona'][:80]}…")
    print(f"  Example: {profile['example_paragraphs'][0][:100]}…\n")

## 2 · Topic research agent

The researcher takes a keyword and produces a comprehensive brief:
simulated competitor analysis, keyword suggestions, content gaps, and
audience questions. We demonstrate on three topics.

In [ ]:
# Simulated SERP database
SERP_DB: dict[str, dict[str, Any]] = {
    "kubernetes autoscaling": {
        "people_also_ask": [
            "How does Kubernetes horizontal pod autoscaler work?",
            "What is the difference between HPA and VPA?",
            "How to set up cluster autoscaler?",
            "What metrics should I use for autoscaling?",
            "Is Kubernetes autoscaling cost-effective?",
        ],
        "competitors": [
            {"title": "K8s Autoscaling Deep Dive", "word_count": 2200,
             "h2_count": 6, "angle": "Technical HPA tutorial", "domain_authority": 72},
            {"title": "Complete Guide to K8s Scaling", "word_count": 1800,
             "h2_count": 5, "angle": "Beginner overview", "domain_authority": 65},
            {"title": "Autoscaling Best Practices 2025", "word_count": 1500,
             "h2_count": 4, "angle": "Production tips", "domain_authority": 58},
        ],
        "related_keywords": [
            "horizontal pod autoscaler", "vertical pod autoscaler",
            "cluster autoscaler", "KEDA", "custom metrics autoscaling",
        ],
        "monthly_volume": 8100,
        "difficulty": 45,
    },
    "content marketing strategy": {
        "people_also_ask": [
            "How do I create a content marketing strategy?",
            "What are the 5 pillars of content marketing?",
            "How much does content marketing cost?",
            "What tools are best for content marketing?",
            "How to measure content marketing ROI?",
        ],
        "competitors": [
            {"title": "Content Strategy Framework 2025", "word_count": 3000,
             "h2_count": 8, "angle": "Comprehensive framework", "domain_authority": 85},
            {"title": "Content Marketing for Startups", "word_count": 1600,
             "h2_count": 5, "angle": "Budget approach", "domain_authority": 52},
            {"title": "B2B Content Strategy Guide", "word_count": 2400,
             "h2_count": 7, "angle": "Enterprise focus", "domain_authority": 78},
        ],
        "related_keywords": [
            "content calendar", "editorial strategy", "content pillars",
            "content distribution", "content repurposing",
        ],
        "monthly_volume": 14800,
        "difficulty": 62,
    },
    "api security best practices": {
        "people_also_ask": [
            "How do I secure a REST API?",
            "What are OWASP API security top 10?",
            "Should I use API keys or OAuth?",
            "How to prevent API rate limiting abuse?",
            "What is API gateway security?",
        ],
        "competitors": [
            {"title": "API Security Checklist 2025", "word_count": 2000,
             "h2_count": 7, "angle": "Checklist format", "domain_authority": 70},
            {"title": "Securing Your REST API", "word_count": 1700,
             "h2_count": 5, "angle": "Tutorial style", "domain_authority": 55},
            {"title": "Enterprise API Security Guide", "word_count": 2800,
             "h2_count": 9, "angle": "Enterprise governance", "domain_authority": 82},
        ],
        "related_keywords": [
            "OAuth 2.0", "API gateway", "rate limiting",
            "JWT authentication", "OWASP API security",
        ],
        "monthly_volume": 6600,
        "difficulty": 38,
    },
}


def research_topic(keyword: str) -> dict[str, Any]:
    """Run topic research on a keyword.

    Parameters
    ----------
    keyword : primary target keyword

    Returns
    -------
    dict – structured research brief with competitor analysis and gaps.
    """
    data = SERP_DB.get(keyword)
    if data is None:
        data = list(SERP_DB.values())[0]
        keyword = list(SERP_DB.keys())[0]

    avg_wc = int(np.mean([c["word_count"] for c in data["competitors"]]))
    avg_h2 = int(np.mean([c["h2_count"] for c in data["competitors"]]))
    angles = [c["angle"] for c in data["competitors"]]

    gaps: list[str] = []
    if not any("cost" in a.lower() or "roi" in a.lower() for a in angles):
        gaps.append("No competitor covers cost/ROI analysis")
    if not any("case study" in a.lower() or "real" in a.lower() for a in angles):
        gaps.append("Missing real-world case studies and examples")
    if avg_wc < 2500:
        gaps.append("Opportunity for more comprehensive coverage")

    return {
        "keyword": keyword,
        "monthly_volume": data["monthly_volume"],
        "keyword_difficulty": data["difficulty"],
        "people_also_ask": data["people_also_ask"],
        "competitor_analysis": data["competitors"],
        "related_keywords": data["related_keywords"],
        "content_gaps": gaps,
        "target_word_count": max(avg_wc + 500, 1500),
        "target_h2_count": avg_h2 + 2,
    }


# Demo: research 3 topics
TOPICS: list[str] = list(SERP_DB.keys())

research_briefs: dict[str, dict[str, Any]] = {}
for topic in TOPICS:
    brief = research_topic(topic)
    research_briefs[topic] = brief
    print(f"━━━ {topic} ━━━")
    print(f"  Volume: {brief['monthly_volume']:,}/mo  |  Difficulty: {brief['keyword_difficulty']}")
    print(f"  Gaps found: {len(brief['content_gaps'])}")
    print(f"  Target: {brief['target_word_count']} words, {brief['target_h2_count']} H2s\n")

## 3 · Brief generator

The brief generator converts research into a detailed writing
specification with H2/H3 outline, key points, and SEO requirements.

In [ ]:
def generate_brief(research: dict[str, Any], intent: str = "informational") -> dict[str, Any]:
    """Generate a content brief from research data.

    Parameters
    ----------
    research : output of research_topic()
    intent : primary search intent type

    Returns
    -------
    dict – structured content brief with outline and SEO reqs.
    """
    kw = research["keyword"]
    paa = research["people_also_ask"]
    related = research["related_keywords"]
    gaps = research["content_gaps"]

    outline: list[dict[str, Any]] = [
        {
            "level": "h2",
            "text": f"What is {kw.title()}?",
            "key_points": [
                f"Define {kw} clearly for the target audience",
                "Explain why it matters in the current landscape",
            ],
            "subsections": [
                {"level": "h3", "text": "Core Concepts"},
                {"level": "h3", "text": "Why It Matters in 2025"},
            ],
        },
        {
            "level": "h2",
            "text": f"How {kw.title()} Works",
            "key_points": [
                f"Technical explanation of {kw} mechanisms",
                f"Relate to {related[0]} and {related[1]}",
            ],
            "subsections": [
                {"level": "h3", "text": f"Understanding {related[0].title()}"},
                {"level": "h3", "text": "Step-by-Step Implementation"},
            ],
        },
    ]

    for q in paa[:3]:
        outline.append({
            "level": "h2",
            "text": q.rstrip("?"),
            "key_points": [f"Directly answer: {q}"],
            "subsections": [],
        })

    for gap in gaps[:2]:
        outline.append({
            "level": "h2",
            "text": gap.replace("No competitor covers ", "").replace("Missing ", "").title(),
            "key_points": [f"Fill gap: {gap}"],
            "subsections": [],
        })

    return {
        "target_keyword": kw,
        "search_intent": intent,
        "target_word_count": research["target_word_count"],
        "outline": outline,
        "related_keywords": related,
        "competitor_gaps": gaps,
        "seo_requirements": {
            "keyword_in_title": True,
            "keyword_in_first_100_words": True,
            "keyword_in_at_least_one_h2": True,
            "meta_description": "150-160 chars with keyword and CTA",
            "internal_links": 3,
            "external_links": 2,
        },
    }


# Generate briefs for all 3 topics
content_briefs: dict[str, dict[str, Any]] = {}
for topic in TOPICS:
    brief = generate_brief(research_briefs[topic])
    content_briefs[topic] = brief
    print(f"━━━ {topic} ━━━")
    print(f"  Intent: {brief['search_intent']}  |  Target: {brief['target_word_count']} words")
    print(f"  Outline ({len(brief['outline'])} sections):")
    for sec in brief["outline"]:
        print(f"    H2: {sec['text']}")
        for sub in sec.get("subsections", []):
            print(f"      H3: {sub['text']}")
    print()

## 4 · Article writer

The writer takes a brief and voice profile, then generates a full
800–1,200 word article via OpenAI. We generate the **same topic in both
voices** and compare the output side by side.

In [ ]:
def write_article(
    brief: dict[str, Any],
    voice_name: str,
    client: OpenAI | None = client,
    model: str = MODEL,
) -> dict[str, Any]:
    """Write an article from a brief using a specific brand voice.

    Parameters
    ----------
    brief : content brief from generate_brief()
    voice_name : key into VOICE_LIBRARY
    client : OpenAI client (None → returns placeholder)
    model : model name

    Returns
    -------
    dict with article text, metadata, and generation info.
    """
    voice_prompt = get_voice_prompt(voice_name)

    outline_text = ""
    for sec in brief["outline"]:
        outline_text += f"\n{sec['level'].upper()}: {sec['text']}\n"
        for kp in sec.get("key_points", []):
            outline_text += f"  - {kp}\n"
        for sub in sec.get("subsections", []):
            outline_text += f"  {sub['level'].upper()}: {sub['text']}\n"

    system_prompt = (
        f"You are a professional content writer. Follow the brand voice "
        f"instructions exactly.\n\n{voice_prompt}\n\n"
        f"RULES:\n"
        f"- Write in Markdown with proper heading hierarchy (## for H2, ### for H3)\n"
        f"- Include the target keyword naturally in the first 100 words\n"
        f"- Target {brief['target_word_count']} words\n"
        f"- Use related keywords naturally: {', '.join(brief['related_keywords'][:5])}\n"
        f"- Maintain brand voice consistently throughout"
    )

    user_prompt = (
        f"Write a comprehensive article about '{brief['target_keyword']}'.\n\n"
        f"Search intent: {brief['search_intent']}\n\n"
        f"Follow this outline exactly:\n{outline_text}\n\n"
        f"Competitor gaps to exploit (include these):\n"
        + "\n".join(f"- {g}" for g in brief.get("competitor_gaps", []))
    )

    start = time.time()

    if client is None:
        # Placeholder when no API key
        voice = VOICE_LIBRARY[voice_name]
        sections = []
        for sec in brief["outline"]:
            sections.append(f"## {sec['text']}\n")
            para = voice["example_paragraphs"][0]
            sections.append(
                f"{brief['target_keyword'].title()} is a critical topic. "
                f"{para} This section covers {sec['text'].lower()} in depth "
                f"with practical examples and real-world data points. "
                f"Understanding {brief['target_keyword']} requires attention "
                f"to {', '.join(brief['related_keywords'][:3])}. "
                f"Organizations that master these concepts see measurable "
                f"improvements in efficiency and cost management. "
                f"The key insight is that {sec.get('key_points', ['best practices'])[0].lower()}.\n"
            )
            for sub in sec.get("subsections", []):
                sections.append(f"### {sub['text']}\n")
                sections.append(
                    f"Diving deeper into {sub['text'].lower()}, we find that "
                    f"the fundamentals of {brief['target_keyword']} apply here "
                    f"with some important nuances. {para[:120]}\n"
                )
        article_text = "\n".join(sections)
    else:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.7,
            max_tokens=3000,
        )
        article_text = response.choices[0].message.content or ""

    elapsed = time.time() - start
    word_count = len(article_text.split())

    return {
        "keyword": brief["target_keyword"],
        "voice": voice_name,
        "article": article_text,
        "word_count": word_count,
        "generation_time": round(elapsed, 2),
        "model": model if client else "placeholder",
    }


# Generate articles for the first topic in both voices
demo_topic = TOPICS[0]
articles: dict[str, dict[str, Any]] = {}

for voice in ["technical_b2b", "friendly_consumer"]:
    result = write_article(content_briefs[demo_topic], voice)
    articles[f"{demo_topic}_{voice}"] = result
    print(f"━━━ {demo_topic} × {voice} ━━━")
    print(f"  Words: {result['word_count']}  |  Time: {result['generation_time']}s")
    print(f"  First 200 chars: {result['article'][:200]}…\n")

## 5 · SEO optimizer and evaluator

Post-writing analysis: keyword usage, heading structure, meta
description, readability scores (via textstat), and a composite
quality scorecard.

In [ ]:
def analyze_seo(article: str, keyword: str, target_wc: int = 1500) -> dict[str, Any]:
    """Comprehensive SEO analysis of an article.

    Parameters
    ----------
    article : full article text in Markdown
    keyword : target keyword
    target_wc : target word count

    Returns
    -------
    dict with SEO metrics, checks, and overall score.
    """
    word_count = len(article.split())
    kw_lower = keyword.lower()
    text_lower = article.lower()

    kw_count = text_lower.count(kw_lower)
    kw_density = (kw_count / word_count * 100) if word_count else 0.0

    h2s = re.findall(r"^##\s+(.+)$", article, re.MULTILINE)
    h3s = re.findall(r"^###\s+(.+)$", article, re.MULTILINE)
    kw_in_headings = any(kw_lower in h.lower() for h in h2s)

    first_100 = " ".join(article.split()[:100]).lower()
    kw_in_intro = kw_lower in first_100

    flesch = textstat.flesch_reading_ease(article)
    fog = textstat.gunning_fog(article)
    avg_sentence_len = textstat.avg_sentence_length(article)

    checks = {
        "word_count_adequate": word_count >= target_wc * 0.8,
        "keyword_density_ok": 0.5 <= kw_density <= 2.5,
        "keyword_in_headings": kw_in_headings,
        "keyword_in_intro": kw_in_intro,
        "has_h2_structure": len(h2s) >= 3,
        "has_h3_depth": len(h3s) >= 2,
        "readability_ok": 30 <= flesch <= 80,
    }
    score = sum(checks.values()) / len(checks)

    return {
        "word_count": word_count,
        "keyword_count": kw_count,
        "keyword_density_pct": round(kw_density, 2),
        "h2_count": len(h2s),
        "h3_count": len(h3s),
        "flesch_reading_ease": round(flesch, 1),
        "gunning_fog": round(fog, 1),
        "avg_sentence_length": round(avg_sentence_len, 1),
        "checks": checks,
        "seo_score": round(score, 2),
    }


def evaluate_quality(
    article: str,
    seo_result: dict[str, Any],
    voice_name: str,
) -> dict[str, Any]:
    """Multi-dimensional quality evaluation.

    Parameters
    ----------
    article : article text
    seo_result : output of analyze_seo()
    voice_name : brand voice used

    Returns
    -------
    dict with dimension scores and overall grade.
    """
    voice = VOICE_LIBRARY[voice_name]
    text_lower = article.lower()

    # Depth: approximate by word count, heading depth, and paragraph count
    paragraphs = [p for p in article.split("\n\n") if len(p.strip()) > 50]
    depth_score = min(5, max(1, round(
        (seo_result["word_count"] / 400)
        + (seo_result["h2_count"] / 2)
        + (len(paragraphs) / 4)
    ) // 2))

    # Readability: map Flesch to 1-5
    flesch = seo_result["flesch_reading_ease"]
    readability_score = (
        5 if 50 <= flesch <= 70 else
        4 if 40 <= flesch <= 80 else
        3 if 30 <= flesch <= 90 else
        2 if 20 <= flesch <= 100 else 1
    )

    # Voice adherence: check preferred/avoided vocabulary
    prefer_hits = sum(1 for w in voice["vocabulary"]["prefer"] if w.lower() in text_lower)
    avoid_hits = sum(1 for w in voice["vocabulary"]["avoid"] if w.lower() in text_lower)
    voice_score = min(5, max(1, prefer_hits - avoid_hits + 3))

    # SEO from actual analysis
    seo_score = min(5, max(1, round(seo_result["seo_score"] * 5)))

    # Originality: placeholder heuristic (real = LLM judge)
    originality_score = 3

    dimensions = {
        "depth": {"score": depth_score, "weight": 0.25},
        "originality": {"score": originality_score, "weight": 0.20},
        "readability": {"score": readability_score, "weight": 0.20},
        "seo_compliance": {"score": seo_score, "weight": 0.20},
        "voice_consistency": {"score": voice_score, "weight": 0.15},
    }

    total = sum(
        (d["score"] / 5.0) * d["weight"] for d in dimensions.values()
    )
    grade = (
        "A" if total >= 0.85 else
        "B" if total >= 0.70 else
        "C" if total >= 0.55 else
        "D" if total >= 0.40 else "F"
    )

    return {
        "dimensions": dimensions,
        "total_score": round(total, 3),
        "grade": grade,
    }


# Score articles
for key, art_data in articles.items():
    seo = analyze_seo(art_data["article"], art_data["keyword"])
    quality = evaluate_quality(art_data["article"], seo, art_data["voice"])
    print(f"━━━ {key} ━━━")
    print(f"  SEO score: {seo['seo_score']:.2f}")
    for dim, info in quality["dimensions"].items():
        print(f"  {dim:.<25} {info['score']}/5")
    print(f"  TOTAL: {quality['total_score']:.3f} (Grade {quality['grade']})\n")

## 6 · End-to-end pipeline

Full pipeline on all 3 topics: research → brief → write → optimize →
evaluate → revise. We track score improvement across the revision loop
and print final articles with SEO metadata.

In [ ]:
def run_pipeline(
    keyword: str,
    voice_name: str = "technical_b2b",
    max_revisions: int = 2,
) -> dict[str, Any]:
    """Run the full content engine pipeline.

    Parameters
    ----------
    keyword : target keyword
    voice_name : brand voice to use
    max_revisions : maximum revision iterations

    Returns
    -------
    dict with final article, all scores, and revision history.
    """
    # Stage 1: Research
    research = research_topic(keyword)

    # Stage 2: Brief
    brief = generate_brief(research)

    # Stage 3+: Write → Evaluate → Revise loop
    history: list[dict[str, Any]] = []

    for iteration in range(max_revisions + 1):
        result = write_article(brief, voice_name)
        seo = analyze_seo(result["article"], keyword, brief["target_word_count"])
        quality = evaluate_quality(result["article"], seo, voice_name)

        history.append({
            "iteration": iteration,
            "word_count": result["word_count"],
            "seo_score": seo["seo_score"],
            "quality_score": quality["total_score"],
            "grade": quality["grade"],
        })

        if quality["total_score"] >= 0.75:
            break

    meta_description = (
        f"{keyword.title()}: comprehensive guide covering key concepts, "
        f"best practices, and real-world examples. Learn everything you need."
    )[:160]

    return {
        "keyword": keyword,
        "voice": voice_name,
        "article": result["article"],
        "word_count": result["word_count"],
        "seo_analysis": seo,
        "quality": quality,
        "meta_description": meta_description,
        "revision_history": history,
    }


# Run pipeline on all 3 topics
pipeline_results: list[dict[str, Any]] = []
voices = ["technical_b2b", "friendly_consumer", "technical_b2b"]

for topic, voice in zip(TOPICS, voices):
    print(f"\n{'='*60}")
    print(f"PIPELINE: {topic} × {voice}")
    print(f"{'='*60}")

    result = run_pipeline(topic, voice)
    pipeline_results.append(result)

    print(f"  Words: {result['word_count']}")
    print(f"  SEO: {result['seo_analysis']['seo_score']:.2f}")
    print(f"  Quality: {result['quality']['total_score']:.3f} ({result['quality']['grade']})")
    print(f"  Meta: {result['meta_description']}")
    print(f"  Revisions: {len(result['revision_history'])}")

    # Show revision history
    for h in result["revision_history"]:
        marker = "✓" if h["quality_score"] >= 0.75 else "→"
        print(f"    {marker} iter {h['iteration']}: quality={h['quality_score']:.3f}")

    # Show first 300 chars of final article
    print(f"\n  Article preview:\n  {result['article'][:300]}…")

In [ ]:
# Summary table
print("\n" + "=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)
print(f"{'Topic':<35} {'Voice':<20} {'Words':>6} {'SEO':>5} {'Grade':>6}")
print("-" * 70)
for r in pipeline_results:
    print(
        f"{r['keyword']:<35} {r['voice']:<20} "
        f"{r['word_count']:>6} {r['seo_analysis']['seo_score']:>5.2f} "
        f"{r['quality']['grade']:>6}"
    )

## Exercises

1. **Third brand voice** — Add a "thought leadership" voice profile to
   `VOICE_LIBRARY` with provocative, visionary tone. Run the pipeline
   and compare output quality to the existing two voices.
2. **Revision loop improvement** — Modify `run_pipeline` to pass
   specific revision instructions (from `evaluate_quality`) back into
   the writer's user prompt on each iteration. Measure whether
   targeted feedback produces faster score convergence.
3. **ChromaDB voice retrieval** — Store all example paragraphs in
   ChromaDB. At writing time, retrieve the 3 most similar past
   paragraphs as few-shot context instead of using static examples.

## Takeaways

* Brand voice is **parameterised**, not hard-coded — the same pipeline
  serves multiple audiences.
* Research → brief → write is the core loop; SEO optimization and
  quality eval are **post-processing** steps.
* The **revision loop** with targeted feedback improves output quality
  measurably across iterations.
* End-to-end pipelines need **instrumentation** — track word count,
  SEO score, and quality grade at every stage.

## What's next

In **03 — Evaluate** we'll run a rigorous evaluation across all
generated articles: LLM-as-judge for depth and originality, SEO
compliance audits, brand voice consistency, readability benchmarks,
and cost analysis.

---

## References

1. OpenAI — "Prompt Engineering Guide" (2024)
   https://platform.openai.com/docs/guides/prompt-engineering
2. Moz — "The Beginner's Guide to SEO" (2024)
   https://moz.com/beginners-guide-to-seo
3. Content Marketing Institute — "B2B Content Marketing Report" (2024)
   https://contentmarketinginstitute.com/research/